# Ch 32 부록 — diffusion LM 붕괴의 진짜 범인을 직접 찾기: 작은 모델엔 작은 vocab

> 본 챕터(Ch 32)에서 작은 diffusion LM이 vocab **2048** 로 영어 동화를 생성하는 걸 봤습니다. 그런데 처음엔 일반 BERT 토크나이저(WordPiece, vocab **30522**)를 그대로 썼다가, 모델이 유니그램(`. the the and`)으로 **붕괴** 했습니다. 모델 구조도 데이터도 그대로인데요. 이 부록은 그 **예전 실패를 직접 복구** 하며, *작은 모델엔 왜 작은 vocab이 필요한지* 를 손으로 확인합니다.

**이 부록에서 직접 경험할 것**
1. 💥 **붕괴 재현** — WordPiece 30522 로 작은 모델을 학습 → 유니그램으로 무너짐
2. 🔬 **진단** — 파라미터 배분을 찍어 보면, 임베딩 테이블이 모델의 **약 70%** 를 잡아먹음
3. 🛠️ **수정** — TinyStories 에 BPE 2048 을 직접 학습 → 임베딩 14% → 본체 용량 회복 → 생성 살아남

**환경**: Google Colab **T4 GPU**. **예상 소요**: 약 18분 (데이터 + 학습 2회를 경량 step으로).

> ⚠️ *경량 재현* 입니다. 완전한 coherent 생성은 30000 step(본 챕터)이 필요하고, 이 부록은 짧은 step으로 *붕괴(유니그램) vs 정상(단어)* 의 **방향** 만 비교합니다.

## 0. 환경 셋업

In [ ]:
%pip install -q -U transformers tokenizers datasets accelerate

import math, time, warnings, torch
import torch.nn.functional as F
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
STEPS = 8000          # 경량 — 붕괴 vs 정상의 방향만 (본 챕터는 30000)
print("device:", device, "| fp16:", USE_FP16, "| steps:", STEPS)

## 1. 데이터 (TinyStories) + 공통 함수

데이터·모델 구조(hidden 256/4L)·loss(`1/t` 시간가중)는 두 실험이 똑같이 씁니다. *유일하게 바꾸는 건 토크나이저(vocab)* 입니다.

In [ ]:
from datasets import load_dataset
from transformers import BertConfig, BertForMaskedLM, Trainer, TrainingArguments

raw_train = load_dataset("roneneldan/TinyStories", split="train[:30000]")
raw_val   = load_dataset("roneneldan/TinyStories", split="validation[:500]")
BLOCK = 128

def prep_with(tok):
    '''주어진 토크나이저로 토큰화 + BLOCK 단위 group.'''
    def tok_fn(b): return {"input_ids": tok(b["text"], add_special_tokens=False)["input_ids"]}
    def group(b):
        cat = sum(b["input_ids"], []); n = (len(cat)//BLOCK)*BLOCK
        return {"input_ids": [cat[i:i+BLOCK] for i in range(0, n, BLOCK)]}
    tr = raw_train.map(tok_fn, batched=True, remove_columns=raw_train.column_names).map(group, batched=True)
    va = raw_val.map(tok_fn,   batched=True, remove_columns=raw_val.column_names).map(group, batched=True)
    return tr, va

def new_model(vocab_size):
    cfg = BertConfig(vocab_size=vocab_size, hidden_size=256, num_hidden_layers=4,
                     num_attention_heads=4, intermediate_size=1024,
                     max_position_embeddings=BLOCK, pad_token_id=0)
    return BertForMaskedLM(cfg).to(device)

def embed_share(model):
    '''임베딩 파라미터가 전체에서 차지하는 비율.'''
    emb = model.bert.embeddings.word_embeddings.weight.numel()
    return emb, model.num_parameters(), emb / model.num_parameters()

class DiffusionCollator:
    def __init__(self, tok, eps=0.02, seed=42):
        self.mask_id = tok.mask_token_id; self.eps = eps
        self.gen = torch.Generator().manual_seed(seed)
    def __call__(self, ex):
        ids = torch.tensor([e["input_ids"] for e in ex], dtype=torch.long); B, L = ids.shape
        t = torch.rand(B, generator=self.gen) * (1.0 - self.eps) + self.eps
        mask = torch.rand(B, L, generator=self.gen) < t.unsqueeze(1)
        no = ~mask.any(1)
        if no.any(): mask[no, torch.randint(0, L, (int(no.sum()),), generator=self.gen)] = True
        inp = ids.clone(); inp[mask] = self.mask_id
        lab = ids.clone(); lab[~mask] = -100
        return {"input_ids": inp, "attention_mask": torch.ones(B, L, dtype=torch.long),
                "labels": lab, "t": t}

class DiffusionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        t = inputs["t"]; labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        B, L, V = out.logits.shape
        per = F.cross_entropy(out.logits.view(-1, V), labels.view(-1),
                              ignore_index=-100, reduction="none").view(B, L)
        loss = ((per.sum(1)/L) / t.to(per.dtype)).mean()
        return (loss, out) if return_outputs else loss

def args_for(tag):
    return TrainingArguments(output_dir=f"./out_{tag}", max_steps=STEPS,
        per_device_train_batch_size=64, learning_rate=3e-4, weight_decay=0.01,
        warmup_steps=500, lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
        logging_steps=200, save_strategy="no", report_to="none", label_names=["labels"],
        remove_unused_columns=False, seed=SEED)

@torch.no_grad()
def diffusion_generate(model, tok, length=48, steps=16, temperature=1.0, top_k=50):
    '''전부 [MASK]에서 시작해 steps번 denoise (기본 confidence-remasking 샘플러).'''
    model.eval(); mask_id = tok.mask_token_id
    x = torch.full((1, length), mask_id, dtype=torch.long, device=device)
    for step in range(steps):
        logits = model(input_ids=x).logits[0]; probs = logits.softmax(-1)
        scaled = logits / temperature
        kth = scaled.topk(top_k, -1).values[:, -1, None]
        scaled = scaled.masked_fill(scaled < kth, float("-inf"))
        pred = torch.multinomial(scaled.softmax(-1), 1).squeeze(-1)
        conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
        is_mask = (x[0] == mask_id); xn = torch.where(is_mask, pred, x[0])
        n_remain = int(round(int(is_mask.sum()) * (1 - (step+1)/steps)))
        if n_remain > 0:
            cm = conf.clone(); cm[~is_mask] = float("inf")
            xn[cm.topk(n_remain, largest=False).indices] = mask_id
        x[0] = xn
    return tok.decode(x[0], skip_special_tokens=True)
print("준비 완료")

## 2. 💥 시도 1 — 예전 실패 복구: WordPiece 30522 그대로 쓰기

가장 자연스러운 첫 시도입니다. "diffusion 은 BERT 계열이니, BERT 의 토크나이저(`bert-base-uncased`, WordPiece)를 그대로 쓰면 되겠지." `[MASK]` 도 내장돼 있어 편합니다. 모델은 작게(hidden 256/4L) 두고요.

In [ ]:
from transformers import AutoTokenizer

tok_big = AutoTokenizer.from_pretrained("bert-base-uncased")   # WordPiece, vocab 30522
lm_train_big, lm_val_big = prep_with(tok_big)

torch.manual_seed(SEED)
model_big = new_model(tok_big.vocab_size)
emb, tot, share = embed_share(model_big)
print(f"vocab {tok_big.vocab_size} | 모델 {tot/1e6:.2f}M | 임베딩 {emb/1e6:.2f}M ({share:.0%})")

out_big = DiffusionTrainer(model=model_big, args=args_for("big"),
                           train_dataset=lm_train_big, data_collator=DiffusionCollator(tok_big)).train()
gen_big = diffusion_generate(model_big, tok_big)
print(f"\n[붕괴 시도] train_loss {out_big.training_loss:.3f}  (baseline ln(V)={math.log(tok_big.vocab_size):.2f})")
print(f"생성: {gen_big[:200]!r}")
print("→ 단어가 아니라 고빈도 토큰(구두점·the·and)이 반복되는 유니그램 붕괴입니다.")

## 3. 🔬 진단 — 임베딩이 모델을 잡아먹었다

vocab 이 30522 이고 hidden 이 256 이면, 임베딩 테이블만

$$30522 \times 256 \approx 7.8\text{M}$$

개의 파라미터입니다. 전체가 약 11M 인 작은 모델에서 **임베딩이 약 70%** 를 차지하니, 정작 문맥을 읽고 빈칸을 채우는 Transformer 본체에 쓸 용량이 거의 남지 않습니다. 게다가 출력 softmax 도 매 자리 30522-way 분류라 학습 신호가 넓게 흩어집니다. 위 셀에서 찍은 임베딩 비율(약 70%)이 그 증거입니다. 모델이 *문맥* 이 아니라 *빈도* 만 외운 까닭이지요.

## 4. 🛠️ 수정 — TinyStories에 BPE 2048을 직접 학습

해법은 데이터에 맞는 작은 vocab 을 직접 학습하는 것입니다. TinyStories 코퍼스에 ByteLevel BPE 2048 을 학습하면 임베딩은

$$2048 \times 256 \approx 0.5\text{M}$$

로 줄어, 비중이 70%에서 **14%** 로 내려갑니다. 모델 구조(hidden 256/4L)·step·loss 는 *그대로* 두고, 토크나이저만 바꿉니다.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

def corpus_iter(bs=1000):
    for i in range(0, len(raw_train), bs):
        yield raw_train[i:i+bs]["text"]
_tk = Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
_tk.decoder = decoders.ByteLevel()
_tk.train_from_iterator(corpus_iter(), trainer=trainers.BpeTrainer(
    vocab_size=2048, special_tokens=["[PAD]", "[UNK]", "[MASK]"]))
tok_small = PreTrainedTokenizerFast(tokenizer_object=_tk, pad_token="[PAD]",
                                    unk_token="[UNK]", mask_token="[MASK]")
lm_train_small, lm_val_small = prep_with(tok_small)

torch.manual_seed(SEED)
model_small = new_model(tok_small.vocab_size)
emb, tot, share = embed_share(model_small)
print(f"vocab {tok_small.vocab_size} | 모델 {tot/1e6:.2f}M | 임베딩 {emb/1e6:.2f}M ({share:.0%})")

out_small = DiffusionTrainer(model=model_small, args=args_for("small"),
                             train_dataset=lm_train_small, data_collator=DiffusionCollator(tok_small)).train()
gen_small = diffusion_generate(model_small, tok_small)
print(f"\n[수정] train_loss {out_small.training_loss:.3f}  (baseline ln(V)={math.log(tok_small.vocab_size):.2f})")
print(f"생성: {gen_small[:200]!r}")
print("→ 본체 용량이 살아나 단어·구절이 나옵니다 (8000 step 이라 거칠지만 방향은 분명).")

## 5. 🆚 나란히 비교

In [ ]:
print("=" * 70)
print(f"{'WordPiece 30522 (붕괴)':<30} 임베딩 70% | 생성: {gen_big[:60]!r}")
print(f"{'BPE 2048 (수정)':<30} 임베딩 14% | 생성: {gen_small[:60]!r}")
print("=" * 70)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
e1, t1, s1 = embed_share(model_big); e2, t2, s2 = embed_share(model_small)
ax.bar(["WordPiece\n30522", "BPE\n2048"], [s1, s2], color=["tab:red", "tab:green"], alpha=0.85)
for i, s in enumerate([s1, s2]): ax.text(i, s + 0.01, f"{s:.0%}", ha="center")
ax.set_ylabel("embedding share of params")
ax.set_title("Small model: big vocab eats the body")
ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

## 6. 정리 — 무엇을 배웠나

직접 겪은 순서:

1. **붕괴 재현**: diffusion 이 BERT 계열이라 WordPiece 30522 를 그대로 썼더니, 작은 모델이 유니그램으로 무너짐.
2. **진단**: 임베딩 테이블(30522×256≈7.8M)이 11M 모델의 **약 70%** 를 잡아먹어 본체 용량이 고갈.
3. **수정**: TinyStories 에 BPE 2048 을 직접 학습 → 임베딩 14% → 본체가 문맥을 배울 여력 회복 → 생성이 단어로 살아남.

**핵심 교훈**
- **작은 모델엔 작은 vocab.** 임베딩 테이블은 `vocab × hidden` 이라, vocab 이 크면 작은 모델에서 파라미터의 대부분을 임베딩이 가져갑니다. 정작 *문맥 추론* 을 하는 본체엔 용량이 안 남습니다.
- "전 세계 영어를 위한 큰 사전" 이 아니라 **"이 데이터를 위한 작은 사전"** 을 직접 학습하는 게, 작은 from-scratch 모델의 핵심 레시피입니다.
- 같은 모델·같은 step·같은 loss 인데 *토크나이저 하나* 로 붕괴와 정상이 갈렸습니다. 본 챕터가 vocab 2048 을 쓴 이유가 바로 이것입니다.